In [1]:
'''Gemma 4 | Budget-Aware Coding Agent - Patch Finalization

Original notebook and budget strategy: matterhorn3838, Gemma 4 SuperAgent.
Source: https://www.kaggle.com/code/matterhorn3838/gemma-4-superagent
The source credits the official sample_submission prompt and community
investigations of graph tools. Thank you to the author and contributors.

Parent: our Efficient Navigation notebook, reported public score 0.22.
This candidate has not been scored. It retains the parent's navigation and
adds one mechanical patch check before submission: changed/new files,
protected paths, git diff --check, and Python syntax without imports.
The check reports problems and never deletes or restores files itself.
Model, sampling, tool declarations and evaluation limits are unchanged.
All cells are Python; run them in order.

The 8-minute task cap is not a guarantee of the global 12-hour limit.
This notebook packages the agent without running model inference or
submitting to Kaggle.
'''

from pathlib import Path
import hashlib
import os
import zipfile
import subprocess
import sys
import tempfile
import yaml

OUTPUT_DIR = Path(os.environ.get('GEMMA_OUTPUT_ROOT', '/kaggle/working'))
if not OUTPUT_DIR.is_dir() and 'GEMMA_OUTPUT_ROOT' not in os.environ:
    OUTPUT_DIR = Path.cwd() / 'gemma4_output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = OUTPUT_DIR / 'submission.zip'


In [2]:
'''Add patch finalization to the 0.22 agent without changing its budgets.'''

FINALIZE_SCRIPT = r'''import subprocess
import sys
from pathlib import Path

def git(*args):
    return subprocess.check_output(['git', *args], timeout=10)

def paths(data):
    return [name.decode('utf-8', 'surrogateescape') for name in data.split(b'\0') if name]

new = paths(git('ls-files', '--others', '--exclude-standard', '-z'))
subprocess.run(['git', 'add', '-N', '.'], check=True, timeout=10)
changed = paths(git('diff', 'HEAD', '--name-only', '-z'))
protected = []
errors = []
for name in changed:
    path = Path(name)
    if (any(part.lower() in ('tests', 'test', 'testing') for part in path.parts)
        or path.name.startswith('test_') or path.name.endswith('_test.py')
        or path.name in ('conftest.py', 'pytest.ini', 'pyproject.toml', 'tox.ini', 'setup.cfg')):
        protected.append(name)
    if path.suffix == '.py' and (path.is_file() or path.is_symlink()):
        if path.is_symlink():
            errors.append(name + ': symlink not compiled')
            continue
        try:
            compile(path.read_bytes(), name, 'exec')
        except (SyntaxError, ValueError, OSError) as exc:
            errors.append(name + ': ' + type(exc).__name__ + ': ' + str(exc)[:200])
check = subprocess.run(['git', 'diff', '--check', 'HEAD'], capture_output=True, text=True, timeout=10)
print('CHANGED_FILES:', changed)
print('NEW_FILES:', new)
print('PROTECTED_FILES:', protected)
print('SYNTAX_ERRORS:', errors[:10])
print('DIFF_CHECK_EXIT:', check.returncode)
print((check.stdout + check.stderr)[:1500])
print('NO_SOURCE_PATCH' if not changed else 'PATCH_PRESENT')
sys.exit(1 if errors or protected or check.returncode or not changed else 0)
'''
FINALIZE_COMMAND = "python3 - <<'PATCH_CHECK'\n" + FINALIZE_SCRIPT + '\nPATCH_CHECK'

SYSTEM_PROMPT = """You are an autonomous software engineer. Your job: make the smallest correct change to the SOURCE code in /workspace so that the hidden tests for the issue below pass. The hidden tests were written from the issue text, so the exact names, messages, types and behaviours in the issue are the specification.

# HARD LIMITS
- 28 tool calls and 8 minutes in total. When either runs out, the working tree is graded as it is. An edit that exists counts, even if you never call `submit_patch`. An unedited tree always scores zero.
- Never create, modify or delete test files (`test_*.py`, `*_test.py`, anything under `tests/`). This is an automatic failure.
- Never run a full test suite (bare `pytest`, `pytest .`, `unittest discover`). Always name one test file, and add `-k` when you can.
- Never look outside /workspace. All dependencies are already installed.
- Never use `search_similar_code`, `get_code_neighbors` or `get_code_subgraph`. They return nothing useful.

# CALL PLAN
- Calls 1-8: locate and read. Calls 9-12: edit. You MUST have edited a source file by call 12. If you are unsure, make your best-guess edit in the most likely place now. A wrong edit can still be corrected. Pure reading cannot score.
- Calls 13-20: verify and correct.
- Call 20 onward: no new exploration. Check, fix, submit.
- Always finish with the mechanical patch check in step 5 followed by submit_patch.

# PROCEDURE

## 1. Extract the specification (no tool call)
From the issue, list for yourself: the exact file, function, class or CLI name; the exact error message and exception type; the expected return value or output; the inputs that trigger the bug; every edge case mentioned (None, empty, negative, unicode, duplicates, defaults). Treat every concrete string and name in the issue as a literal that must appear exactly in your code.

## 2. Locate
- Use exact paths from the issue directly. When searches are necessary, combine independent searches for the issue's exact symbols, their callers and relevant test paths into one run_command. Use git grep with fixed-string patterns for literal identifiers and git ls-files for tracked file paths; each result should show its file path and line number where applicable.
- Batch only searches whose inputs are already known. If the next search depends on the first result, inspect that result before searching again. Limit each search to about 20 matching lines and keep total output short; never dump whole files or the repository tree.
- Once the target is located, read the relevant implementation plus one relevant caller or test case. When both paths and line ranges are known, read these small regions in one run_command using sed with explicit line ranges, or use read_file for a single region. Aim for about 40 lines per region and at most three regions per command. Include enough surrounding code to understand signatures, defaults and error handling.
- For a bug: find where the wrong behaviour is produced and use the caller or test to understand the contract before editing.
- For a feature: find the closest existing sibling (a similar function, option, class, or doc example) and copy its pattern, including registration points such as __all__, option lists, dispatch tables and imports.
- Reuse paths and test names already discovered. Do not spend another tool call repeating a search that already answered the question.
- If the issue includes a code snippet that reproduces the problem, you may run it once with python -c (one call, never create a file) to see the real error.

## 3. Edit
- Use edit_file for changes to existing code. Copy old_string exactly from the source, including whitespace, and choose a short unique match. Use write_file only for a genuinely new source file that the issue requires.
- If edit_file reports no match or multiple matches, read the exact source region and retry with a smaller unique snippet. Never repeat an identical failed edit. After two matching failures, use one inline Python script through run_command with a quoted heredoc; assert that the old text occurs exactly once before replacing it and writing the file. Preserve unrelated content and do not rewrite the file from memory.
- Fix the cause, not the symptom. Do not special-case the example from the issue.
- Change only what the issue requires. Keep existing signatures, defaults, and backward compatibility unless the issue says otherwise. Match the style, naming and error-handling idioms of the surrounding code.
- Use the exact exception class and exact message text from the issue. If the issue only describes the error, reuse the wording style of neighbouring error messages.
- Handle every edge case you listed in step 1.
- For documentation-code tasks (for example FastAPI), edit the executable code under `docs_src/`.

## 4. Verify (at most 2 calls)
- Find the test file with `find tests -name "*keyword*.py"`, then run `timeout 150 python -m pytest tests/path/test_x.py -x -q -k keyword`. Use `python3 -m unittest tests.test_x` for unittest projects.
- Failures that come from missing fixtures, missing data, or import errors unrelated to your change are pre-existing. Ignore them. Never repair tests or the environment.
- If a test related to your change fails, read the traceback, fix the source once, and rerun once. If no test can run, do one `python -c` check or `python -m py_compile` on the edited file.

## 5. Final check and submit
- Reserve one run_command for this mechanical check before submission. It includes newly added files in the diff using git add -N, checks whitespace and Python syntax without imports, and reports protected files. It never deletes or restores files:

PATCH_FINALIZER_COMMAND

- Review CHANGED_FILES and NEW_FILES against the issue. A required new module is valid; do not delete it just because it is new. Remove only temporary artifacts you created. Undo only your own unintended protected-file edits. The check's protected-path list is conservative; configuration-only edits are not the implementation fix.
- If the check reports a specific syntax or diff error, fix the source and rerun the check only if time remains. Do not start new exploration or rewrite a correct patch just to satisfy the checker. A check timeout is inconclusive. Submit the best justified source patch before the session expires.
- Call submit_patch as the final intended tool action. The harness may terminate the loop immediately, so complete all checks and edits before calling it.

# DECISION RULES
- Ambiguous issue: choose the most literal reading of the issue text and act on it. Do not ask questions and do not stall.
- Several possible locations: pick the one where the wrong behaviour is produced, not where it is merely observed.
- Search returned nothing: try one different specific string, then edit your best candidate anyway.
- Stuck after a failed fix: make a different, smaller change. Do not repeat the same edit.
- Never conclude that nothing needs to change. Every task requires a non-empty source patch.
"""

SYSTEM_PROMPT = SYSTEM_PROMPT.replace('PATCH_FINALIZER_COMMAND', FINALIZE_COMMAND)

FILES = {
    "agent.yaml": """name: swe_agent
model: gemma-4-31b-it-qat-w4a16-ct
instruction: !include prompts/system.md
tools:
  - run_command
  - read_file
  - edit_file
  - write_file
  - get_status
  - submit_patch
  - get_code_neighbors
  - search_similar_code
  - get_code_subgraph
generate_content_config: !include configs/sampling.yaml
""",
    "configs/sampling.yaml": """temperature: 0.2
top_p: 0.95
max_output_tokens: 8192
thinking_config:
  thinking_budget: 4096
  include_thoughts: true
""",
    "eval_config.yaml": """evaluation:
  timeout_seconds: 240
  max_tool_calls: 28
  max_time_minutes: 8
  max_turns: 80
""",
    "prompts/system.md": SYSTEM_PROMPT,
}


In [3]:
'''Check the single-agent configuration and local includes.'''

class IncludeLoader(yaml.SafeLoader):
    pass

def include(loader, node):
    relative = loader.construct_scalar(node)
    assert relative in FILES, relative
    return FILES[relative] if relative.endswith('.md') else yaml.safe_load(FILES[relative])

IncludeLoader.add_constructor('!include', include)
agent = yaml.load(FILES['agent.yaml'], Loader=IncludeLoader)
evaluation = yaml.safe_load(FILES['eval_config.yaml'])['evaluation']
assert agent['model'] == 'gemma-4-31b-it-qat-w4a16-ct'
assert agent['instruction'] == SYSTEM_PROMPT
assert agent['generate_content_config'] == yaml.safe_load(FILES['configs/sampling.yaml'])
assert set(agent['tools']) == {
    'run_command', 'read_file', 'edit_file', 'write_file', 'get_status',
    'submit_patch', 'get_code_neighbors', 'search_similar_code', 'get_code_subgraph',
}
assert evaluation == {
    'timeout_seconds': 240, 'max_tool_calls': 28,
    'max_time_minutes': 8, 'max_turns': 80,
}
assert set(FILES) == {'agent.yaml', 'configs/sampling.yaml', 'eval_config.yaml', 'prompts/system.md'}
print('Configuration checks passed: one agent, 8 minutes, 28 calls, 240-second timeout.')
print('Global runtime and leaderboard score require competition evaluation.')


'''Exercise the finalizer on disposable repositories without touching task files.'''

def check_finalizer():
    with tempfile.TemporaryDirectory(prefix='gemma4_finalizer_') as folder:
        repo = Path(folder)
        def git(*args):
            return subprocess.run(['git', *args], cwd=repo, check=True, capture_output=True)
        def run_check():
            return subprocess.run([sys.executable, '-c', FINALIZE_SCRIPT], cwd=repo, capture_output=True, text=True, timeout=30)
        git('init', '-q')
        (repo / 'source.py').write_text('value = 1\n')
        (repo / 'deleted.py').write_text('value = 1\n')
        git('add', '.')
        git('-c', 'user.name=Notebook Check', '-c', 'user.email=check@example.invalid', 'commit', '-qm', 'fixture')
        (repo / 'source.py').write_text('value = 2\n')
        (repo / 'deleted.py').unlink()
        added = repo / 'new module.py'
        added.write_bytes(b'# coding: latin-1\nlabel = "caf\xe9"\n')
        result = run_check()
        assert result.returncode == 0, result.stdout + result.stderr
        assert 'new module.py' in result.stdout and added.exists()
        added.write_text('def broken(\n')
        result = run_check()
        assert result.returncode == 1 and 'SyntaxError' in result.stdout, result.stdout + result.stderr
        added.write_text('value = 3\n')
        (repo / 'source.py').write_text('value = 2 \n')
        result = run_check()
        assert result.returncode == 1 and 'trailing whitespace' in result.stdout, result.stdout + result.stderr
        (repo / 'source.py').write_text('value = 2\n')
        protected = repo / 'pytest.ini'
        protected.write_text('[pytest]\n')
        result = run_check()
        assert result.returncode == 1 and "PROTECTED_FILES: ['pytest.ini']" in result.stdout, result.stdout + result.stderr
        assert protected.exists() and added.exists()
        assert not list(repo.rglob('*.pyc'))
        protected.unlink()
        result = run_check()
        assert result.returncode == 0, result.stdout + result.stderr

check_finalizer()
print('Finalizer self-check passed: new/deleted files, encoding, syntax, whitespace and protected paths.')


Configuration checks passed: one agent, 8 minutes, 28 calls, 240-second timeout.
Global runtime and leaderboard score require competition evaluation.
Finalizer self-check passed: new/deleted files, encoding, syntax, whitespace and protected paths.


In [4]:
'''Package the submission files in a reproducible ZIP archive.'''

with zipfile.ZipFile(ZIP_PATH, 'w') as archive:
    for relative, content in sorted(FILES.items()):
        entry = zipfile.ZipInfo(relative, date_time=(1980, 1, 1, 0, 0, 0))
        entry.compress_type = zipfile.ZIP_DEFLATED
        entry.external_attr = 0o100644 << 16
        archive.writestr(entry, content.encode('utf-8'))
with zipfile.ZipFile(ZIP_PATH) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(FILES)
    for relative, content in FILES.items():
        assert archive.read(relative) == content.encode('utf-8')
print('Submission:', ZIP_PATH)
print('SHA-256:', hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest())
print('Archive files:', sorted(FILES))
print('Official model compilation and leaderboard scoring were not run.')


Submission: /Users/flexonafft/MLKaggleTasks/competitions/public-comp/gemma-4-developer-agent/artifacts/patch_finalization/submission.zip
SHA-256: 0c680a270c9b6e09126ff081709f5ae32436721c6033a0b08058eb4fcb37485e
Archive files: ['agent.yaml', 'configs/sampling.yaml', 'eval_config.yaml', 'prompts/system.md']
Official model compilation and leaderboard scoring were not run.
